In [11]:
from ollama import chat
from ollama import ChatResponse
from loguru import logger
import time

In [12]:
QUESTION = "Describe the people in the image. Extract the number of people and their characteristics. Provide a detailed description of each person, including their appearance, clothing, and any notable features. Additionally, mention the overall mood or atmosphere conveyed by the people in the image."

# VLM-based Reflection Agent

In [ ]:
VLM_MODEL = 'qwen3-vl:2b'

# Reflection Agent

In [14]:
from ollama import chat


class ReflectionAgent:

    BASE_SYSTEM = (
        "You are a helpful assistant. "
        "Be concise, accurate, and clear."
    )

    GENERATOR_SYSTEM = (
        BASE_SYSTEM
        + "\nTask: Answer the user's request directly."
    )

    CRITIC_SYSTEM = (
        BASE_SYSTEM
        + "\nTask: Critique the assistant response. "
          "Identify concrete weaknesses and missing details."
    )

    REVISER_SYSTEM = (
        BASE_SYSTEM
        + "\nTask: Improve the response using the critique. "
          "Return only the revised response."
    )

    EVALUATOR_SYSTEM = (
        BASE_SYSTEM
        + "\nTask: Evaluate response quality. "
          "Summarize strengths and weaknesses."
    )

    def __init__(self, model: str):
        self.model = model
        self.user_context = []

    def add_message(self, content: str, images: list | None = None):

        msg = {
            "role": "user",
            "content": content,
        }

        if images:
            msg["images"] = images

        self.user_context.append(msg)

    def _chat(self, system_prompt, messages):

        response = chat(
            model=self.model,
            messages=[
                {
                    "role": "system",
                    "content": system_prompt,
                },
                *messages,
            ],
        )

        return response["message"]["content"]

    def generate_response(self):

        response = self._chat(
            self.GENERATOR_SYSTEM,
            self.user_context,
        )

        return response

    def critique_response(self, response: str):

        critique = self._chat(
            self.CRITIC_SYSTEM,
            [
                {
                    "role": "user",
                    "content": (
                        "Critique this response:\n\n"
                        f"{response}"
                    ),
                }
            ],
        )

        return critique

    def revise_response(
        self,
        original_response: str,
        critique: str,
    ):

        revised = self._chat(
            self.REVISER_SYSTEM,
            [
                {
                    "role": "user",
                    "content": (
                        f"""
Original response:

{original_response}

Critique:

{critique}

Rewrite the response.
"""
                    ),
                }
            ],
        )

        return revised

    def evaluate_response(self, response: str):

        evaluation = self._chat(
            self.EVALUATOR_SYSTEM,
            [
                {
                    "role": "user",
                    "content": (
                        "Evaluate this response:\n\n"
                        f"{response}"
                    ),
                }
            ],
        )

        return evaluation

    def run(self):

        initial = self.generate_response()

        critique = self.critique_response(
            initial
        )

        revised = self.revise_response(
            initial,
            critique,
        )

        evaluation = self.evaluate_response(
            revised
        )

        return {
            "initial_response": initial,
            "critique": critique,
            "final_response": revised,
            "evaluation": evaluation,
        }


In [ ]:
if __name__ == "__main__":

    agent = ReflectionAgent(VLM_MODEL)

    agent.add_message(content=QUESTION, images=["imgs/car.jpeg"])

    start_time = time.time()

    result = agent.run()

    elapsed_time = time.time() - start_time

    logger.info(f"Elapsed time: {elapsed_time:.2f} seconds")


2026-06-13 18:32:42.160 | INFO     | __main__:<module>:16 - Elapsed time: 55.21 seconds
2026-06-13 18:32:42.161 | INFO     | __main__:<module>:18 - Initial response:
There are two people in the image.  

- **Person on the left**: An older individual with gray hair, wearing a dark blue jacket and dark pants. They sit with their arms crossed, appearing relaxed.  
- **Person on the right**: An older individual wearing a red hat, a brown jacket, and a scarf. They are seated next to a blue blanket with patterns, also appearing relaxed.  

The overall atmosphere conveyed by the scene is calm and comfortable, suggesting a peaceful, leisurely moment spent outdoors. The setting (a beach with water and mountains) and the cozy setup inside the van (with a blanket and curtains) contribute to a sense of tranquility and relaxation.
2026-06-13 18:32:42.162 | INFO     | __main__:<module>:19 - Critique:
### Critique of the Assistant Response  
**Major Weakness:**  
❌ **Misinterpreting the Task:** The a

In [16]:
logger.info("Initial response:\n" + result["initial_response"])

2026-06-13 18:33:53.588 | INFO     | __main__:<module>:1 - Initial response:
There are two people in the image.  

- **Person on the left**: An older individual with gray hair, wearing a dark blue jacket and dark pants. They sit with their arms crossed, appearing relaxed.  
- **Person on the right**: An older individual wearing a red hat, a brown jacket, and a scarf. They are seated next to a blue blanket with patterns, also appearing relaxed.  

The overall atmosphere conveyed by the scene is calm and comfortable, suggesting a peaceful, leisurely moment spent outdoors. The setting (a beach with water and mountains) and the cozy setup inside the van (with a blanket and curtains) contribute to a sense of tranquility and relaxation.


In [17]:
logger.info("Critique:\n" + result["critique"])

2026-06-13 18:33:55.699 | INFO     | __main__:<module>:1 - Critique:
### Critique of the Assistant Response  
**Major Weakness:**  
❌ **Misinterpreting the Task:** The assistant incorrectly assumed the image *already existed* ("There are two people in the image"). However, the task explicitly required analyzing *what the assistant observed* (the assistant is describing a fictional image). This is a fundamental misstep that invalidates the critique.  

**Concrete Weaknesses:**  
1. **Vague and Inconsistent Details:**  
   - The description of the left person ("older individual with gray hair") lacks specificity (e.g., *how* old? *What is the context of the gray hair?*).  
   - The right person’s clothing is described with minimal context (e.g., "red hat" is not explained as *why* it matters for the atmosphere).  
   - **Missing Critical Detail:** No mention of *how* the two people interact or position themselves relative to the scene (e.g., "the left person sits facing the right person"

In [18]:
logger.info("Final response:\n" + result["final_response"])

2026-06-13 18:33:59.637 | INFO     | __main__:<module>:1 - Final response:
There are two people in the image.  

- **Person on the left**: A middle-aged individual with short gray hair, wearing a dark blue jacket and dark pants. They sit with their arms crossed, leaning slightly toward the right person, suggesting a relaxed but engaged posture.  
- **Person on the right**: An older individual wearing a red hat and a brown jacket, with a scarf draped around their neck. They sit beside a blue blanket with patterns, their legs crossed.  

The scene is set on a beach with calm water and distant mountains. Inside a van with soft lighting, the interior features a blue blanket and curtains, creating a cozy atmosphere.  

The overall atmosphere is peaceful and intimate, conveying a sense of quiet companionship. The blue blanket and red hat contribute to a warm yet tranquil mood, while the mountain backdrop and water reflections enhance the sense of calm. The natural setting (water, mountains) 

In [19]:
logger.info("Evaluation:\n" + result["evaluation"])

2026-06-13 18:34:27.505 | INFO     | __main__:<module>:1 - Evaluation:
### Evaluation of Response  

**Strengths:**  
- **Clear structure:** Well-organized with bullet points for the two individuals and the setting, making the description easy to follow.  
- **Detailed and accurate:** Precise descriptions of clothing, posture, and environmental elements (e.g., "blue blanket," "red hat," "mountain backdrop") are consistent with the scene.  
- **Coherent atmosphere:** Effectively conveys the peaceful, intimate mood through sensory details (e.g., "calm water," "cozy interior").  
- **Logical flow:** Connects elements (e.g., the blue blanket and red hat) to the overall mood without redundancy.  

**Weaknesses:**  
- **Minor inconsistency:** The description of the left person’s posture ("leaning slightly toward the right person") is slightly ambiguous; it could be interpreted as leaning *toward* the right person but not explicitly "leaning toward the right." (This is minor and contextually 